In [4]:
from langchain_community.document_loaders import TextLoader


In [9]:
# 1. Load the document from the file
doc = TextLoader(file_path='diabetes_factsheet.txt')
dia = doc.load()
dia

[Document(metadata={'source': 'diabetes_factsheet.txt'}, page_content='Diabetes Factsheet\n\nDiabetes is a chronic disease characterized by elevated blood glucose levels. The main types are type 1, type 2, and gestational diabetes.\n\n**Management**: Diabetes management includes a balanced diet, regular exercise, blood glucose monitoring, and medication (such as insulin or oral hypoglycemic agents). Complications can be prevented with good control.\n\n**Symptoms**: Increased thirst, frequent urination, unexplained weight loss, and fatigue.\n\n**Prevention**: Maintain a healthy weight, exercise regularly, and have regular health checkups.\n\nFor personalized guidance, consult a healthcare professional.\n')]

In [ ]:
# 2. Break this doc into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 50, chunk_overlap = 10)
chunks = splitter.split_documents(dia)
chunks

In [20]:
# 3. Perform embedding of these chunks
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')
chunk_lists = [chunk.page_content for chunk in chunks]
embedded_chunks = embeddings.embed_documents(chunk_lists)
len(embedded_chunks)

C:\Users\thedo\AppData\Local\Temp\ipykernel_23856\2809916690.py:6: RuntimeWarning: coroutine 'OpenAIEmbeddings.aembed_documents' was never awaited
  embedded_chunks = embeddings.embed_documents(chunk_lists)


19

In [24]:
# 4. Now Lets store this embedded data to the
from langchain_community.vectorstores import FAISS

vector_storage = FAISS.from_documents(chunks, embeddings)

# 5. Test

question = 'what is diabeties'


res = vector_storage.similarity_search(question, k= 4)


for doc in res:
    print(doc.page_content)

retriever = vector_storage.as_retriever(search_kwargs={'k': 3})

diabetes.
Diabetes is a chronic disease characterized by
Diabetes Factsheet
**Management**: Diabetes management includes a


In [ ]:
# 6. Finally feed this to an LLM and then fetch the response:

from langchain_openai import OpenAI
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate


model = OpenAI(model= 'gpt-4o-mini')

prompt = '''You are a health care expert 
{context}
Question: {input}
Answer here: 
'''


prompt_template= PromptTemplate(template= prompt, input_variables=['context', 'input'])
chain = create_stuff_documents_chain(model, prompt_template)

retriever_chain = create_retrieval_chain(retriever, combine_docs_chain= chain)

result = retriever_chain.invoke({'input': 'What is diabeties ?'})
print(result['answer'])